# Notebook 03 — Unsupervised Clustering & Topic Modeling
**Project:** TruthLens — Multilingual Fake News Detection  
**Goal:** K-Means clustering, LDA topic modeling, DBSCAN anomaly detection, t-SNE visualization

---

## 1. Import Libraries

In [ ]:
import os
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

plt.style.use('dark_background')
MODELS_DIR   = '../models/'
DATASETS_DIR = '../datasets/'
os.makedirs(MODELS_DIR, exist_ok=True)

print('Libraries imported ✓')

## 2. Load Data & Build TF-IDF

In [ ]:
data_path = f'{DATASETS_DIR}processed_dataset.csv'
if not os.path.exists(data_path):
    raise FileNotFoundError('Run Notebook 01 first!')

df = pd.read_csv(data_path).dropna(subset=['cleaned_text', 'label'])
df['cleaned_text'] = df['cleaned_text'].astype(str)
df['label'] = df['label'].astype(int)

# Sample for speed — use up to 30K articles for unsupervised
if len(df) > 30000:
    df = df.sample(30000, random_state=42).reset_index(drop=True)
    print(f'Sampled 30,000 articles for unsupervised analysis')
else:
    print(f'Using all {len(df):,} articles')

texts = df['cleaned_text'].tolist()
labels = df['label'].values

# Build TF-IDF
print('\nBuilding TF-IDF matrix...')
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1,1),
                              sublinear_tf=True, min_df=5, max_df=0.9)
X_tfidf = vectorizer.fit_transform(texts)
print(f'TF-IDF shape: {X_tfidf.shape}')

# Reduce dimensions with TruncatedSVD for clustering
print('Reducing dimensions with TruncatedSVD (100 components)...')
svd = TruncatedSVD(n_components=100, random_state=42)
X_reduced = svd.fit_transform(X_tfidf)
print(f'Variance explained: {svd.explained_variance_ratio_.sum():.2%}')
print(f'Reduced shape: {X_reduced.shape}')

## 3. K-Means Clustering — Elbow Curve

In [ ]:
print('Running K-Means for k = 3, 5, 8, 10, 15...')
print('(This takes ~5-8 minutes)')

k_values = [3, 5, 8, 10, 15]
inertias = []
silhouette_scores = []
km_models = {}

for k in k_values:
    print(f'  Fitting k={k}...', end=' ')
    km = KMeans(n_clusters=k, init='k-means++', max_iter=300,
                n_init=10, random_state=42)
    cluster_labels = km.fit_predict(X_reduced)
    inertias.append(km.inertia_)

    sil = silhouette_score(X_reduced, cluster_labels,
                           sample_size=min(3000, len(X_reduced)))
    silhouette_scores.append(sil)
    km_models[k] = {'model': km, 'labels': cluster_labels}
    print(f'Inertia: {km.inertia_:.0f}  Silhouette: {sil:.4f}')

best_k_idx = np.argmax(silhouette_scores)
best_k = k_values[best_k_idx]
print(f'\nBest k = {best_k} (silhouette score = {silhouette_scores[best_k_idx]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('K-Means Analysis', fontsize=14, fontweight='bold')

# Elbow curve
axes[0].plot(k_values, inertias, 'o-', color='#7c6af7', linewidth=2, markersize=8)
axes[0].axvline(x=best_k, color='#f7b731', linestyle='--', alpha=0.7, label=f'Best k={best_k}')
axes[0].set_title('Elbow Curve (Inertia)', fontsize=12)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].legend()

# Silhouette scores
axes[1].plot(k_values, silhouette_scores, 's-', color='#3ecf8e', linewidth=2, markersize=8)
axes[1].axvline(x=best_k, color='#f7b731', linestyle='--', alpha=0.7, label=f'Best k={best_k}')
axes[1].set_title('Silhouette Scores', fontsize=12)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}kmeans_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Inspect Cluster Topics

In [ ]:
best_km = km_models[best_k]['model']
best_labels = km_models[best_k]['labels']
feature_names = vectorizer.get_feature_names_out()

# Project cluster centers back to original TF-IDF space
centroids_original = svd.inverse_transform(best_km.cluster_centers_)

print(f'Top words per cluster (k={best_k}):')
print('='*60)
cluster_info = []
for i, center in enumerate(centroids_original):
    top_indices = center.argsort()[-10:][::-1]
    top_words = [feature_names[idx] for idx in top_indices]
    fake_ratio = labels[best_labels == i].mean()
    cluster_info.append({'cluster': i, 'top_words': top_words, 'fake_ratio': fake_ratio})
    print(f'Cluster {i} (fake ratio: {fake_ratio:.1%}): {" | ".join(top_words[:7])}')

print('='*60)
print('Clusters with fake_ratio > 0.5 are predominantly FAKE news clusters')

## 5. LDA Topic Modeling

In [ ]:
print('Running LDA Topic Modeling (10 topics)...')
print('This takes ~5 minutes...')

N_TOPICS = 10

# LDA uses raw counts, not TF-IDF
count_vec = CountVectorizer(max_features=10000, min_df=5, max_df=0.90,
                             stop_words='english')
X_counts = count_vec.fit_transform(texts)

lda = LatentDirichletAllocation(
    n_components=N_TOPICS, max_iter=20,
    learning_method='online', learning_offset=50.,
    random_state=42
)
doc_topics = lda.fit_transform(X_counts)
perplexity = lda.perplexity(X_counts)

print(f'LDA complete! Perplexity: {perplexity:.2f} (lower = better)')

# Top words per topic
lda_feature_names = count_vec.get_feature_names_out()
topic_words = {}
print('\nTop words per topic:')
print('='*60)
for idx, topic in enumerate(lda.components_):
    top_indices = topic.argsort()[-12:][::-1]
    words = [lda_feature_names[i] for i in top_indices]
    topic_words[f'Topic {idx+1}'] = words
    print(f'Topic {idx+1:2d}: {" | ".join(words[:8])}')
print('='*60)

In [ ]:
# ── Fake vs Real Topic Distribution ──────────────────────────────────────────
fake_mask = labels == 1
real_mask = labels == 0

topic_comparison = []
for i in range(N_TOPICS):
    fake_avg = doc_topics[fake_mask, i].mean()
    real_avg = doc_topics[real_mask, i].mean()
    topic_comparison.append({
        'Topic': f'Topic {i+1}',
        'Top Words': ', '.join(topic_words[f'Topic {i+1}'][:4]),
        'Avg in FAKE': round(fake_avg, 4),
        'Avg in REAL': round(real_avg, 4),
        'Bias': 'FAKE' if fake_avg > real_avg else 'REAL'
    })

topic_df = pd.DataFrame(topic_comparison).sort_values('Avg in FAKE', ascending=False)
print('Topic distribution — FAKE vs REAL:')
print(topic_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(N_TOPICS)
width = 0.35
ax.bar(x - width/2, topic_df['Avg in FAKE'].values, width, label='FAKE', color='#ff6b6b', alpha=0.85)
ax.bar(x + width/2, topic_df['Avg in REAL'].values, width, label='REAL', color='#3ecf8e', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f'T{i+1}' for i in range(N_TOPICS)])
ax.set_title('Topic Distribution: Fake vs Real News', fontsize=13)
ax.set_xlabel('Topic')
ax.set_ylabel('Average Topic Weight')
ax.legend()
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}lda_topics.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. DBSCAN Anomaly Detection

In [ ]:
print('Running DBSCAN anomaly detection...')

# Use smaller reduced space for DBSCAN speed
svd_small = TruncatedSVD(n_components=30, random_state=42)
X_small = svd_small.fit_transform(X_tfidf)

db = DBSCAN(eps=0.5, min_samples=5, n_jobs=-1)
db_labels = db.fit_predict(X_small)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_anomalies = (db_labels == -1).sum()
anomaly_ratio = n_anomalies / len(db_labels)

print(f'DBSCAN clusters found: {n_clusters}')
print(f'Anomalous articles:    {n_anomalies:,} ({anomaly_ratio:.1%} of corpus)')

# What label do the anomalies have?
anomaly_labels = labels[db_labels == -1]
if len(anomaly_labels) > 0:
    print(f'Anomalies that are FAKE: {(anomaly_labels==1).sum():,} ({(anomaly_labels==1).mean():.1%})')
    print(f'Anomalies that are REAL: {(anomaly_labels==0).sum():,} ({(anomaly_labels==0).mean():.1%})')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
categories = ['Normal articles', 'Anomalies (DBSCAN -1)']
counts = [len(db_labels) - n_anomalies, n_anomalies]
axes[0].bar(categories, counts, color=['#3ecf8e', '#ff6b6b'])
axes[0].set_title('DBSCAN: Normal vs Anomalous', fontsize=12)
axes[0].set_ylabel('Count')
for i, v in enumerate(counts):
    axes[0].text(i, v + 50, f'{v:,}', ha='center')

if len(anomaly_labels) > 0:
    axes[1].pie([((anomaly_labels==1).sum()), ((anomaly_labels==0).sum())],
                labels=['Fake anomalies', 'Real anomalies'],
                autopct='%1.1f%%', colors=['#ff6b6b', '#3ecf8e'])
    axes[1].set_title('Anomaly Label Breakdown', fontsize=12)

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}dbscan_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. t-SNE Visualization

In [ ]:
print('Running t-SNE (this takes ~5-10 mins)...')
print('Sampling 3,000 articles for speed...')

N_TSNE = 3000
idx = np.random.choice(len(X_reduced), N_TSNE, replace=False)
X_sample = X_reduced[idx]
labels_sample = labels[idx]
cluster_sample = best_labels[idx]

tsne = TSNE(n_components=2, perplexity=30, random_state=42,
            n_iter=1000, learning_rate='auto', init='pca')
X_2d = tsne.fit_transform(X_sample)
print(f't-SNE done! KL divergence: {tsne.kl_divergence_:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('t-SNE Visualization of News Articles', fontsize=15, fontweight='bold')

# Plot 1: colored by FAKE/REAL label
colors_label = ['#3ecf8e' if l == 0 else '#ff6b6b' for l in labels_sample]
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=colors_label, alpha=0.4, s=8)
axes[0].set_title('Colored by Label (Green=Real, Red=Fake)', fontsize=12)
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')
from matplotlib.lines import Line2D
legend_elements = [Line2D([0],[0], marker='o', color='w', markerfacecolor='#3ecf8e', label='Real', markersize=8),
                   Line2D([0],[0], marker='o', color='w', markerfacecolor='#ff6b6b', label='Fake', markersize=8)]
axes[0].legend(handles=legend_elements)

# Plot 2: colored by cluster
cmap = cm.get_cmap('tab20', best_k)
scatter = axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=cluster_sample,
                           cmap='tab20', alpha=0.4, s=8)
axes[1].set_title(f'Colored by K-Means Cluster (k={best_k})', fontsize=12)
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
plt.colorbar(scatter, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}tsne_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('t-SNE plot saved → models/tsne_visualization.png')

# Save t-SNE data as JSON for Flask dashboard
tsne_data = {
    'x': X_2d[:, 0].tolist(),
    'y': X_2d[:, 1].tolist(),
    'label': labels_sample.tolist(),
    'cluster': cluster_sample.tolist()
}
with open(f'{MODELS_DIR}tsne_results.json', 'w') as f:
    json.dump(tsne_data, f)
print('t-SNE data saved → models/tsne_results.json')

## 8. Save Unsupervised Models

In [ ]:
# Save K-Means
with open(f'{MODELS_DIR}kmeans_model.pkl', 'wb') as f:
    pickle.dump({'model': best_km, 'svd': svd, 'best_k': best_k,
                 'cluster_info': cluster_info}, f)
print('K-Means model saved ✓')

# Save LDA
with open(f'{MODELS_DIR}lda_model.pkl', 'wb') as f:
    pickle.dump({'model': lda, 'vectorizer': count_vec,
                 'topic_words': topic_words, 'n_topics': N_TOPICS}, f)
print('LDA model saved ✓')

# Save topic words as JSON for Flask dashboard
with open(f'{MODELS_DIR}topic_words.json', 'w') as f:
    json.dump(topic_words, f)
print('Topic words saved ✓')

print('\n' + '='*50)
print('NOTEBOOK 03 COMPLETE')
print('='*50)
print('Unsupervised analysis done:')
print(f'  K-Means: best k={best_k}, silhouette={silhouette_scores[best_k_idx]:.4f}')
print(f'  LDA:     {N_TOPICS} topics, perplexity={perplexity:.2f}')
print(f'  DBSCAN:  {n_anomalies:,} anomalies detected ({anomaly_ratio:.1%})')
print(f'  t-SNE:   {N_TSNE} articles visualized')
print('\nFiles in models/:')
for fname in sorted(os.listdir(MODELS_DIR)):
    size = os.path.getsize(f'{MODELS_DIR}{fname}')
    print(f'  {fname}  ({size/1024:.1f} KB)')
print('\nNext step → python app.py to launch the Flask web app!')